# 02 — Preprocessing Pipeline Impact

**Tujuan**: benchmark IR metric (MAP via BM25) untuk setiap variant preprocessing.
Untuk rubric Preprocessing 15% — wajib show empirical impact.

## Setup

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, '../backend')

import json
import pandas as pd
from rank_bm25 import BM25Okapi

from app.preprocessing import PipelineConfig, PreprocessingPipeline
from app.evaluation import average_precision, mean_average_precision

## Load Data

In [ ]:
with open('../data/raw/mamikos_real_v2.jsonl', encoding='utf-8') as f:
    raw_listings = [json.loads(line) for line in f if line.strip()]
raw_listings = [r for r in raw_listings if (r.get('deskripsi') or '').strip()]
print(f'Loaded {len(raw_listings)} real listings')

with open('../eval/queries.json', encoding='utf-8') as f:
    queries = json.load(f)['queries']

with open('../eval/ground_truth.csv') as f:
    import csv
    gt_rows = list(csv.DictReader(f))

# Build relevant set per query (relevance >= 1)
relevant_per_query = {}
for row in gt_rows:
    qid = row['query_id']
    if int(row['relevance']) >= 1:
        relevant_per_query.setdefault(qid, set()).add(row['doc_id'])

## Benchmark Function

In [ ]:
def benchmark_config(config: PipelineConfig, label: str) -> dict:
    """Run pipeline, build BM25, compute MAP."""
    pipeline = PreprocessingPipeline(config)
    
    # Preprocess corpus
    docs = [pipeline.process(l['deskripsi']).processed for l in raw_listings]
    doc_ids = [l['id'] for l in raw_listings]
    
    # Build BM25
    tokenized = [d.split() for d in docs]
    bm25 = BM25Okapi(tokenized, k1=1.5, b=0.75)
    
    # Run queries
    predicted_per_query = {}
    for q in queries:
        q_processed = pipeline.process(q['query']).processed
        scores = bm25.get_scores(q_processed.split())
        # Top-10
        import numpy as np
        top_idx = np.argsort(-scores)[:10]
        predicted_per_query[q['id']] = [doc_ids[i] for i in top_idx]
    
    map_score = mean_average_precision(predicted_per_query, relevant_per_query)
    return {'config': label, 'map': map_score}

## Run Benchmark per Variant

Toggle individual stages off untuk lihat impact masing-masing.

WARNING: tiap variant butuh ~30 detik (preprocessing + BM25 build).

In [ ]:
variants = [
    ('BASELINE (no preprocessing)', PipelineConfig(
        strip_html=False, normalize_whitespace=False, extract_prices=False,
        lowercase=False, apply_jargon_dict=False, correct_spelling=False,
        tokenize=False, remove_stopwords=False, stem=False,
    )),
    ('+ lowercase only', PipelineConfig(
        strip_html=True, normalize_whitespace=True, extract_prices=False,
        lowercase=True, apply_jargon_dict=False, correct_spelling=False,
        tokenize=True, remove_stopwords=False, stem=False,
    )),
    ('+ jargon dict (108 entries)', PipelineConfig(
        strip_html=True, normalize_whitespace=True, extract_prices=False,
        lowercase=True, apply_jargon_dict=True, correct_spelling=False,
        tokenize=True, remove_stopwords=False, stem=False,
    )),
    ('+ spelling correction', PipelineConfig(
        strip_html=True, normalize_whitespace=True, extract_prices=False,
        lowercase=True, apply_jargon_dict=True, correct_spelling=True,
        tokenize=True, remove_stopwords=False, stem=False,
    )),
    ('+ stopword removal', PipelineConfig(
        strip_html=True, normalize_whitespace=True, extract_prices=False,
        lowercase=True, apply_jargon_dict=True, correct_spelling=True,
        tokenize=True, remove_stopwords=True, stem=False,
    )),
    ('FULL pipeline (+ stem)', PipelineConfig()),
]

results = []
for label, cfg in variants:
    print(f'Running: {label}')
    results.append(benchmark_config(cfg, label))

results_df = pd.DataFrame(results)
results_df

## Visualize Impact

In [ ]:
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-whitegrid')

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(results_df['config'], results_df['map'], color='#2563eb')
ax.set_xlabel('MAP (BM25, top-10)')
ax.set_title('Preprocessing Pipeline Stage Impact pada BM25 MAP')
for i, v in enumerate(results_df['map']):
    ax.text(v + 0.005, i, f'{v:.4f}', va='center')
plt.tight_layout()
plt.savefig('charts/02_preprocessing_impact.png', dpi=100, bbox_inches='tight')
plt.show()

## Discussion (template untuk laporan)

1. **BASELINE** (raw text) sebagai reference
2. **+ lowercase** memberikan boost karena query case insensitive
3. **+ jargon dict** memberikan boost paling besar — handle abbreviation domain (kmd, ac, gdg meneng)
4. **+ spelling correction** lebih berdampak di data real (typo asli pemilik: 'kreta', 'belakanga')
5. **+ stopword** boost kecil — kos/kamar terlalu sering muncul
6. **+ stem** kompromise — beberapa kata jadi terlalu pendek (gedong neng) tapi consolidate morphology

**Anti-pattern surface**: lowercase sebelum extract harga akan break `Rp 850.000` matching, walaupun regex case-insensitive masih handle.